In [1]:
import pandas as pd

unplanned_df = pd.read_csv("../Data/cleaned/unplanned_visits_cleaned.csv")
print("Shape:", unplanned_df.shape)
print("\nColumns:")
for col in unplanned_df.columns:
    print(f"  {col}")

Shape: (67046, 19)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  measure_id
  measure_name
  compared_to_national
  denominator
  score
  lower_estimate
  higher_estimate
  number_of_patients
  number_of_patients_returned
  start_date
  end_date


In [2]:
print("unique measures:", unplanned_df["measure_id"].nunique())
print("\nmeasure list:")
print(unplanned_df[["measure_id", "measure_name"]].drop_duplicates().to_string())

unique measures: 14

measure list:
           measure_id                                                                             measure_name
0         EDAC_30_AMI                                           Hospital return days for heart attack patients
1          EDAC_30_HF                                          Hospital return days for heart failure patients
2          EDAC_30_PN                                              Hospital return days for pneumonia patients
3          Hybrid_HWR                                 Hybrid Hospital-Wide All-Cause Readmission Measure (HWR)
4               OP_32            Rate of unplanned hospital visits after colonoscopy (per 1,000 colonoscopies)
5           OP_35_ADM              Rate of inpatient admissions for patients receiving outpatient chemotherapy
6            OP_35_ED  Rate of emergency department (ED) visits for patients receiving outpatient chemotherapy
7               OP_36                     Ratio of unplanned hospital visits 

In [4]:
unplanned_df["score"] = unplanned_df["score"].replace("Not Available", None)
unplanned_df["score"] = pd.to_numeric(unplanned_df["score"], errors="coerce")

unplanned_df["compared_to_national"] = unplanned_df["compared_to_national"].replace("Not Available", None)

score_pivot = unplanned_df.pivot_table(
    index="facility_id",
    columns="measure_id",
    values="score",
    aggfunc="first"
).reset_index()
score_pivot.columns.name = None

compared_pivot = unplanned_df.pivot_table(
    index="facility_id",
    columns="measure_id",
    values="compared_to_national",
    aggfunc="first"
).reset_index()
compared_pivot.columns.name = None
compared_pivot.columns = ["facility_id"] + [f"{col}_compared" for col in compared_pivot.columns[1:]]

hospital_info = unplanned_df[["facility_id", "facility_name", "address",
                               "citytown", "state", "zip_code",
                               "countyparish", "telephone_number"]].drop_duplicates("facility_id")

# merge
df_final = hospital_info.merge(score_pivot, on="facility_id", how="outer")
df_final = df_final.merge(compared_pivot, on="facility_id", how="outer")

print("final shape:", df_final.shape)
print("duplicate facility_ids:", df_final["facility_id"].duplicated().sum())

final shape: (4789, 36)
duplicate facility_ids: 0


In [5]:
df_final.to_csv("../Data/cleaned/unplanned_visits_cleaned.csv", index=False)
print("unplanned_visits saved")

unplanned_visits saved
